## Dependencies

In [1]:
import warnings
warnings.simplefilter("ignore", UserWarning)

In [45]:
import sys
sys.path.append('../')

%load_ext autoreload
%autoreload 2

from libs import ios
from libs import vis
from libs import constants
from libs.bibs import aps
from libs.inference import evaluation
from libs.utils import characternames_handler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Total combinations of features: 1023


## Setup

In [14]:
VERSION = 3
DIR_EVAL = f'../../results/v{VERSION}/gender/predictions_ch/best_models/eval_best_results_<MODEL>.csv'
DIR_PRED = f'../../results/v{VERSION}/gender/predictions_ch/best_models/rs<RS>/'

FN_APS_DEMOGRAPHICS = '/data/datasets/APS/aps_data_processing/shares/aps-2022_25-10-01_e59319256d553035c40188457bb16edbfe02da04.zip' #aps_20240130_2e52fdd7260ea462878821948a2a463ed9acb58a.zip'
FN_GT_ANNOTATIONS = f'../../data/v{VERSION}/annotations/gender/ground_truth.csv'

MODELS = ['regression', 'classification']

In [4]:
vis.sns_paper_style()

## Data

## Our Predictions & GT

In [11]:
df_gt

,gt,single_char,ngender_r,ngender_n,gender_n,gender_w,gender_nw,mean_ngender_rnw,mean_ngender_rn,mean_ngender_rw,mean_ngender_nw,ml_sample
givenname,,,,,,,,,,,,
东,0.166667,True,0.15675,0.451906,male,unknown,male,NaN,0.304328,NaN,NaN,train
习,0.312500,True,0.34495,0.460552,male,unknown,male,NaN,0.402751,NaN,NaN,train
仅,0.458333,True,0.60040,0.545362,female,unknown,female,NaN,0.572881,NaN,NaN,test
县,0.354167,True,0.13905,0.552551,female,unknown,female,NaN,0.345801,NaN,NaN,train
园,0.625000,True,0.78480,0.535775,female,unknown,female,NaN,0.660288,NaN,NaN,train
...,...,...,...,...,...,...,...,...,...,...,...,...
静平,0.479167,False,0.66210,0.538443,female,male,neutral,0.400181,0.600271,0.331050,0.269221,train
颖伶,0.833333,False,0.90300,0.647493,female,female,female,0.850164,0.775246,0.951500,0.823746,train
颖弢,0.625000,False,0.50435,0.389434,male,male,male,0.297928,0.446892,0.252175,0.194717,train


In [13]:
df_gender = ios.pd.DataFrame()
for model in MODELS:
    
    # best overall per model type
    path = DIR_EVAL.replace('<MODEL>', model)
    df_summary = ios.load_csv(path, index_col=0).query("best_overall == True").iloc[0]
    rs = df_summary.rs
    
    # GT
    df_gt = ios.load_csv(ios.path_join(ios.os.path.dirname(df_summary.fn), constants.FN_GT_TRAN_TEST_SAMPLE))
    cols_gt = [c for c in df_gt.columns if c in ['givenname', 'ngender_mean', 'gender_mean', 'ml_sample']]
    df_gt = df_gt[cols_gt].set_index('givenname')
    df_gt.rename(columns={'ngender_mean': 'gt', 'gender_mean': 'gt'}, errors='ignore', inplace=True)

    # all predicitons (all names)
    kind = '_reg_' if model == 'regression' else '_class_'
    path = DIR_PRED.replace('<RS>', str(rs))
    pattern = f'*{kind}*.csv'
    files = ios.listdir(path, pattern, 'csv')
    
    if len(files) == 0:
        print(f'No files found for {model} in {path}')
        continue

    fn = files[0]
    tmp = ios.load_csv(fn)
    assert tmp.shape[0] == tmp.givenname.nunique()
    
    # add gt info
    tmp = tmp.set_index('givenname')
    tmp = tmp.join(df_gt, how='left')
    print(f'{model.title()} | all names with gender: {tmp.shape} | names in gt: {tmp['gt'].notna().sum()}')

    # final df
    cols = [c for c in tmp.columns if c in ['pred_code', 'pred_label', 'pred', 'gt', 'ml_sample']]
    df_gender = tmp[cols].copy() if df_gender.empty else df_gender.join(tmp[cols[:-2]], how='inner')

    
print(df_gender.shape)

[INFO] Found 1 files in ../../results/v3/gender/predictions_ch/best_models/rs2/.
Regression | all names with gender: (46144, 11) | names in gt: 166
[INFO] Found 1 files in ../../results/v3/gender/predictions_ch/best_models/rs2/.
Classification | all names with gender: (46144, 11) | names in gt: 166
(46144, 5)


In [6]:

df_gender.loc[:, 'py_givenname'] = df_gender.apply(lambda row: ''.join(characternames_handler.pinyin_givename(row.name)), axis=1)
df_gender.head(2)

,pred,gt,ml_sample,pred_code,pred_label,py_givenname
givenname,,,,,,
㑇,0.347726,NaN,NaN,0,male,zhou
㑊,0.347726,NaN,NaN,0,male,yi


In [7]:
df_gt = df_gender.query("gt.notna()").copy()
df_gt.sample(2)

,pred,gt,ml_sample,pred_code,pred_label,py_givenname
givenname,,,,,,
国绛,0.374900,0.270833,train,1,neutral,guojiang
丙辰,0.266888,0.166667,test,0,male,bingchen


## Cultural Consensys Theory

In [46]:

df_annotations = ios.load_csv(FN_GT_ANNOTATIONS)
df_annotations.drop(columns=['givenname','single_char', 'round'], inplace=True)

# X: N x M matrix (N = number of annotators, M = number of names)
X = df_annotations.to_numpy().T
X.shape


(11, 166)

In [49]:
c = evaluation.Consensus(X)
results = c.bootstrap_cct(B=2000)
results

{'ratio_median': 11.273571710702102,
 'ratio_ci_2.5': 8.408208271650555,
 'ratio_ci_97.5': 14.739343939982716,
 'var1_median': 0.6743402156984242,
 'var1_ci_2.5': 0.6237899592079172,
 'var1_ci_97.5': 0.7191098416118001,
 'ratios_all': array([10.38989274, 11.71733978, 12.35188004, ..., 12.52682349,
         9.36786808, 12.44747786])}

In [50]:
icc = c.intraclass_correlation_coefficient()
icc

,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,0.622378,19.129645,165,1660,9.663258e-282,"[0.57, 0.68]"
1,ICC2,Single random raters,0.622683,19.591576,165,1650,2.498067e-286,"[0.57, 0.68]"
2,ICC3,Single fixed raters,0.628273,19.591576,165,1650,2.498067e-286,"[0.57, 0.68]"
3,ICC1k,Average raters absolute,0.947725,19.129645,165,1660,9.663258e-282,"[0.94, 0.96]"
4,ICC2k,Average random raters,0.947789,19.591576,165,1650,2.498067e-286,"[0.94, 0.96]"
5,ICC3k,Average fixed raters,0.948958,19.591576,165,1650,2.498067e-286,"[0.94, 0.96]"


## APS

In [51]:
_df_aps_authors = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/authors.csv')
_df_aps_authors_names = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/author_names.csv')
_df_aps_genders = ios.read_csv_from_zip(FN_APS_DEMOGRAPHICS, 'data/genders.csv')

df_aps = ios.pd.merge(_df_aps_authors, _df_aps_authors_names, how='inner', on='id_author', suffixes=("_a","_n"))
df_aps = ios.pd.merge(df_aps, _df_aps_genders, how='left', on='id_gender')
df_aps = df_aps[['id_author','id_author_name','name','gender','gender_nq', 'disambiguated']] # NOTE: it contains disambiguated True and False

print(f"APS authors: {_df_aps_authors.shape}")
print(f"APS authors names: {_df_aps_authors_names.shape}")
print(f"APS genders: {_df_aps_genders.shape}")
print(f"APS all authors merged: {df_aps.shape}")

APS authors: (874556, 4)
APS authors names: (948771, 4)
APS genders: (3, 2)
APS all authors merged: (948771, 6)


In [52]:
df_aps_clean = aps.clean_aps_names(df_aps)
df_aps_clean_ch_py = aps.detect_pinyin_and_chinese_names(df_aps_clean)
df_aps_clean_ch_py.shape

(874556, 21)

In [53]:
df_aps_clean_ch_py.head(2)

,id_author,alternative_names,n_alternative_names,genders,genders_nq,gender_clean,gender_nq_clean,longest_name,longest_name_clean,givenname_clean,...,is_pinyin_givenname,is_pinyin_familyname,is_pinyin,ch_name,ch_splitname,ch_familyname,ch_givenname,ch_translated_name_in_pinyin,ch_translated_familyname_in_pinyin,ch_translated_givenname_in_pinyin
0,0,[S. Megtert],1,[unknown],[-],unknown,unknown,S. Megtert,s. megtert,s.,...,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1,"[Aleksa Bjeli, A. Bjeli]",2,"[male, male]","[-, -]",male,unknown,Aleksa Bjeli,aleksa bjeli,aleksa,...,False,False,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Comparison

In [55]:
df_aps_clean_ch_py.query("ch_givenname.notna() and is_pinyin == True").shape

(1073, 21)

In [61]:
', '.join(df_aps_clean_ch_py.query("ch_givenname.notna()").sample(20).sort_values('ch_givenname').ch_givenname.unique())

'二军, 仁新, 兆华, 兴涛, 创业, 均仁, 巍, 建成, 敏良, 昕, 松, 林港, 林透, 炜平, 玲, 立杰, 聪博, 阳平, 雷, 龙'

In [36]:
df_aps_gt = df_aps_clean_ch_py.query("is_pinyin == True and ch_name.notna() and ch_givenname in @df_gt.index")[['givenname_clean', 'ch_givenname', 'gender_clean', 'gender_nq_clean']].copy()
df_aps_gt.shape

(5, 4)

In [37]:
df_aps_gt

,givenname_clean,ch_givenname,gender_clean,gender_nq_clean
1696,qiong,琼,male,unknown
20603,yaping,亚平,male,unknown
25636,dong,东,male,unknown
37026,fan,帆,male,unknown
67113,yuan,园,male,gm


In [40]:
df_comparison = df_gender.join(df_aps_gt.set_index('ch_givenname').rename(columns={'gender_clean': 'karimi', 'gender_nq_clean': 'burkik'}).drop(columns=['givenname_clean']), how='inner')
df_comparison.loc[:, ['barolo','ours']] = [None, None]
df_comparison

,pred,gt,ml_sample,pred_code,pred_label,py_givenname,karimi,burkik,barolo,ours
东,0.272237,0.166667,train,0,male,dong,male,unknown,None,None
园,0.619605,0.625000,train,1,neutral,yuan,male,gm,None,None
帆,0.376853,0.458333,train,1,neutral,fan,male,unknown,None,None
琼,0.679497,0.812500,train,2,female,qiong,male,unknown,None,None
亚平,0.450442,0.395833,train,1,neutral,yaping,male,unknown,None,None


In [ ]:
# CHECK: Cultural Consensus Theory (CCT)